# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a full example for loading and exploring the [FAIR^2 mass spectrometry dataset](https://doi.org/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library and referencing all entities by their `@id` fields.

### Dataset Source
The dataset source is specified via a Croissant schema URL (JSON-LD).

In [ ]:
# Ensure the latest `mlcroissant` is installed
!pip install -q mlcroissant>=1.0

## 1. Data Loading
Load the Croissant dataset metadata and explore its basic information using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

dataset = mlc.Dataset(croissant_url)

meta = dataset.metadata
print(f"Dataset name: {meta.name}")
print(f"Description: {meta.description}\n")
print(f"Published: {getattr(meta, 'datePublished', 'N/A')}")
print(f"Version: {getattr(meta, 'version', 'N/A')}")
print(f"License: {getattr(meta, 'license', 'N/A')}")

## 2. Data Overview
Review available record sets, their fields and columns, with focus on `@id` references. All further analyses reference entities by their unique `@id`.

> Note: The dataset uses Croissant RecordSets and Fields. Use `.record_sets` and `.fields` attributes.

In [ ]:
# List record sets and their fields/columns by @id
print("Available Record Sets (by @id):")
for rs in dataset.record_sets:
    print(f"  - RecordSet @id: {rs['@id']}  (name: {rs.get('name','N/A')})")
    print("    Columns/Fields:")
    for fld in rs.get('field', []):
        if isinstance(fld, dict):
            print(f"      > {fld.get('@id','N/A')}: {fld.get('name','N/A')}")
        else:
            print(f"      > {fld}")
    print()
print("")

# Optionally: List all available columns by @id
print("All columns by @id:")
for rs in dataset.record_sets:
    for col in rs.get('column', []):
        if isinstance(col, dict):
            print(f"  - {col.get('@id','N/A')} (name: {col.get('name','N/A')})")
        else:
            print(f"  - {col}")

## 3. Data Extraction
Load tabular data from a selected record set into a DataFrame. All references are by `@id`.

Identify one or more record sets suitable for tabular analysis (e.g., the main protein abundance record set and its fields).

In [ ]:
# Select record set(s) for extraction by @id (from previous overview)
# Adjust this list as needed for other record sets
record_sets = [
    rs['@id'] for rs in dataset.record_sets
]

dataframes = {}
for record_set_id in record_sets:
    # Extract records for each record set by @id
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for RecordSet {record_set_id}, shape: {df.shape}")

# Choose the main record set for preview (often there is an obvious 'main' one)
main_record_set = record_sets[0] if len(record_sets) > 0 else None
if main_record_set:
    print("\nColumns in main record set:")
    print(dataframes[main_record_set].columns.tolist())
    dataframes[main_record_set].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps to the main DataFrame:
- Filtering records based on a numeric field (e.g. normalized abundance, peptide count, molecular weight).  
- Normalizing numeric data.  
- Grouping by categorical field (e.g., protein accession or modification type).
All references use field `@id` names as columns.

In [ ]:
# Pick a numeric field and a categorical field (by column name / @id)
df = dataframes[main_record_set]

# Example: Find numeric fields
print("Possible numeric fields (columns):")
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        print(f"  - {col}")

# Example: Use the first numeric column, or choose one by name
numeric_field = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
        break
if numeric_field is None:
    print("No numeric field available.")
else:
    print(f"\nAnalyzing numeric field: {numeric_field}")
    threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records where {numeric_field} > {threshold:.3f} (total {len(filtered_df)})")

    # Normalize the numeric column
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to pick a categorical column
    group_field = None
    for col in df.columns:
        # Heuristic: Pick first non-numeric, non-index, non-accession column
        if not pd.api.types.is_numeric_dtype(df[col]) and col != numeric_field and 'accession' in col.lower():
            group_field = col
            break
    # If not found, pick first object dtype
    if not group_field:
        for col in df.columns:
            if df[col].dtype == object and col != numeric_field:
                group_field = col
                break

    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name=f"{numeric_field}_mean")
        print(f"\nGrouped by {group_field} (mean of {numeric_field}):")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields using matplotlib or seaborn.

Below: Histogram of the numeric field and (optionally) a group-by plot.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True, color='skyblue')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    if group_field is not None:
        plt.figure(figsize=(8, 4))
        top_groups = df[group_field].value_counts().nlargest(10).index
        sns.boxplot(
            data=df[df[group_field].isin(top_groups)],
            x=group_field,
            y=numeric_field
        )
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field} by {group_field} (top 10 groups)")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
- We loaded the FAIR^2 protein abundance dataset via Croissant schema, listing all record sets and fields by `@id`.
- We explored numeric and categorical fields using their `@id` (column name) and performed standard data filtering and normalization.
- Data visualizations summarized main value distributions for protein measurement fields.